# 01 · Data, splits and obfuscated test sets

**Runs on CPU in about a minute.** This notebook:

1. loads the four public datasets and checks their columns,
2. cleans them (English only, no empty rows, no duplicates) and builds leak-free splits,
3. shows each obfuscation family on a benign sentence,
4. builds the shared evaluation bundle: every test sample × {clean + 6 obfuscation families},
   plus its normalised version (hardening method 1),
5. measures how well the normaliser restores obfuscated text, and how much it alters clean text.

Every later notebook reads the files saved here, so all detectors see identical inputs.

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
# os.environ["PI_SMOKE"] = "1"   # uncomment for a 5-minute synthetic test of the whole pipeline

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda v: f"{v:.3f}")

from pi_hardening import config
print("smoke mode:", config.SMOKE, "| data:", config.DATA_DIR)

smoke mode: False | data: C:\Users\admin\PycharmProjects\prompt-injection-hardening\data\processed


In [2]:
from pi_hardening import data, evaluate
from pi_hardening.perturb import perturb, DESCRIPTIONS
from pi_hardening.normalize import Normalizer
TABLES = config.RESULTS_DIR / "tables"; TABLES.mkdir(exist_ok=True)
if config.SMOKE:
    from pi_hardening.smoke import prepare_smoke_models
    prepare_smoke_models()

## 1. Raw datasets
Check that each dataset has a text column and a label column the loader recognises.

In [3]:
raw = pd.concat([data.describe(repo) for repo in config.DATASETS.values()], ignore_index=True)
raw

,dataset,split,rows,columns
0,deepset/prompt-injections,train,546,"text, label"
1,deepset/prompt-injections,test,116,"text, label"
2,Lakera/gandalf_ignore_instructions,train,777,"text, similarity"
3,Lakera/gandalf_ignore_instructions,validation,111,"text, similarity"
4,Lakera/gandalf_ignore_instructions,test,112,"text, similarity"
5,alespalla/chatbot_instruction_prompts,test,64511,"response, prompt"
6,jackhhao/jailbreak-classification,train,1044,"prompt, type"
7,jackhhao/jailbreak-classification,test,262,"prompt, type"
8,leolee99/NotInject,NotInject_one,113,"prompt, word_list, category"
9,leolee99/NotInject,NotInject_two,113,"prompt, word_list, category"


How each raw label value is mapped to the binary label. Every row must say *attack* or *benign*; an `UNMAPPED` value must be added to `POSITIVE` or `NEGATIVE` in `data.py` before continuing.

In [4]:
labels = pd.concat([data.label_values(config.DATASETS[k]) for k in ["deepset", "jackhhao"]], ignore_index=True)
labels.to_csv(TABLES / "label_mapping.csv", index=False)
labels

,dataset,split,column,value,count,maps_to
0,deepset/prompt-injections,train,label,0,343,benign (0)
1,deepset/prompt-injections,train,label,1,203,attack (1)
2,deepset/prompt-injections,test,label,1,60,attack (1)
3,deepset/prompt-injections,test,label,0,56,benign (0)
4,jackhhao/jailbreak-classification,train,type,jailbreak,527,attack (1)
5,jackhhao/jailbreak-classification,train,type,benign,517,benign (0)
6,jackhhao/jailbreak-classification,test,type,jailbreak,139,attack (1)
7,jackhhao/jailbreak-classification,test,type,benign,123,benign (0)


## 2. Cleaning and splits

* **deepset/prompt-injections** mixes English and German; German rows are removed so that all
  sets are English (the obfuscation families target Latin script).
* **Lakera/gandalf_ignore_instructions** contains only attacks and **alespalla/chatbot_instruction_prompts**
  only benign prompts (a seeded sample of 1,000 from its test split), so they get a fixed label.
* Duplicates are removed inside each dataset, and every later split drops texts already in an
  earlier split (train → val → in-domain test → OOD test), so no test text was seen in training.
* **Train / val** pool deepset's official train split with 85 % of Gandalf + benign chat;
  **in-domain test** is deepset's official test split plus the other 15 %; **OOD test** is the jackhhao
  jailbreak set (balanced, capped); **NotInject** is benign only and measures over-defence.

In [5]:
splits, clean_log, overlap_log = data.build_splits()
data.save_splits(splits)
clean_log.to_csv(TABLES / "cleaning_log.csv", index=False)
clean_log

,dataset,raw,empty,non_english,duplicates,kept
0,deepset,662,0,255,0,407
1,gandalf,1000,0,1,1,998
2,benign_chat,1000,1,1,0,998
3,jackhhao,1306,0,6,20,1280
4,notinject,339,0,0,0,339


In [6]:
overlap_log

,split,removed_overlap_with_earlier_splits
0,val,0
1,test_in,0
2,test_ood,0
3,notinject,0


In [7]:
summary = data.split_summary(splits)
summary.to_csv(TABLES / "split_summary.csv", index=False)
summary

,split,source,n,injection,benign,median_words,p95_words
0,train,benign_chat,721,0,721,11.000,24.000
1,train,deepset,284,107,177,9.000,48.700
2,train,gandalf,721,721,0,9.000,28.000
3,val,benign_chat,127,0,127,11.000,25.000
4,val,deepset,51,19,32,9.000,40.500
5,val,gandalf,127,127,0,9.000,29.800
6,test_in,benign_chat,150,0,150,12.000,23.550
7,test_in,deepset,72,37,35,11.000,58.700
8,test_in,gandalf,150,150,0,11.000,28.100
9,test_ood,jackhhao,400,200,200,105.500,780.000


## 3. The six obfuscation families on a benign sentence

In [8]:
example = "Please summarise the attached quarterly report and translate it into French."
norm = Normalizer()
demo = pd.DataFrame([{"family": f, "what it does": DESCRIPTIONS[f],
                      "obfuscated": perturb(example, f, 0.5, seed=1),
                      "after normaliser": norm(perturb(example, f, 0.5, seed=1))}
                     for f in config.FAMILIES])
demo["visible as"] = demo["obfuscated"].map(ascii)   # makes invisible / look-alike characters visible
demo.to_csv(TABLES / "obfuscation_examples.csv", index=False)
demo[["family", "what it does", "obfuscated", "visible as", "after normaliser"]]

,family,what it does,obfuscated,visible as,after normaliser
0,zero_width,invisible zero-width characters inserted inside words,Please su﻿m​mar⁠ise th⁠e attach​e‌d quar⁠te‍r⁠ly report a‌nd t‍ra‍nslat​e it into Fre​n⁠ch.,'Please su\ufeffm\u200bmar\u2060ise th\u2060e attach\u200be\u200cd quar\u2060te\u200dr\u2060ly report a\u200cnd t\u2...,Please summarise the attached quarterly report and translate it into French.
1,homoglyph,letters swapped for look-alike Cyrillic/Greek/Latin-extended characters,Рӏeaѕe summaгiѕҽ the ɑttаcheԁ ԛuαrtеrӏy report anԁ translate it intо French.,'\u0420\u04cfea\u0455e summa\u0433i\u0455\u04bd the \u0251tt\u0430che\u0501 \u051bu\u03b1rt\u0435r\u04cfy report an\...,Piease summarise the attached quarteriy report and translate it into French.
2,diacritics,combining accent marks added to letters,Please summarise the áttac̃ḧed̀ q̄ua̧r̃ter̄ly report and̄ translate it into Fr̃eņc̈h.,'Please summarise the a\u0301ttac\u0303h\u0308ed\u0300 q\u0304ua\u0327r\u0303ter\u0304ly report and\u0304 translate ...,Please summarise the attached quarterly report and translate it into French.
3,leetspeak,"letters replaced by similar-looking digits (e→3, o→0, …)",Pl3a53 summarise the attached qu4rt3rly report and translate it 1nt0 French.,'Pl3a53 summarise the attached qu4rt3rly report and translate it 1nt0 French.',Please summarise the attached quarterly report and translate it into French.
4,char_spacing,spaces inserted between every letter of a word,P l e a s e s u m m a r i s e t h e attached q u a r t e r l y r e p o r t a n d translate it i n t o French.,'P l e a s e s u m m a r i s e t h e attached q u a r t e r l y r e p o r t a n d translate it i n t o French.',Pleasesummarisethe attached quarterlyreportand translate it into French.
5,punct_split,punctuation (. - _ *) inserted inside words,Please summarise the a.t.tached quarterly repo.r.t and t*r*a*nslate it in.to French.,'Please summarise the a.t.tached quarterly repo.r.t and t*r*a*nslate it in.to French.',Please summarise the attached quarterly report and translate it into French.


## 4. Evaluation bundle

Each test sample (both classes) is obfuscated with every family at a rate of
`config.PERTURB_RATE` (share of eligible words changed). Obfuscating benign samples too lets us
check whether a detector reacts to the obfuscation itself rather than to the attack.

In [9]:
bundle = evaluate.build_eval_bundle(splits)
evaluate.save_bundle(bundle)
bundle.groupby(["eval_set", "family"]).agg(n=("uid", "size"), attacks=("label", "sum"),
                                           obfuscated_share=("changed", "mean")).unstack(0)

n                    attacks                   \
eval_set     notinject test_in test_ood notinject test_in test_ood   
family                                                               
char_spacing       NaN 372.000  400.000       NaN 187.000  200.000   
clean          339.000 372.000  400.000     0.000 187.000  200.000   
diacritics         NaN 372.000  400.000       NaN 187.000  200.000   
homoglyph          NaN 372.000  400.000       NaN 187.000  200.000   
leetspeak          NaN 372.000  400.000       NaN 187.000  200.000   
punct_split        NaN 372.000  400.000       NaN 187.000  200.000   
zero_width         NaN 372.000  400.000       NaN 187.000  200.000   

             obfuscated_share                   
eval_set            notinject test_in test_ood  
family                                          
char_spacing              NaN   1.000    0.998  
clean                   0.000   0.000    0.000  
diacritics                NaN   1.000    0.998  
homoglyph                 NaN   1.000    0.998  
leetspeak                 NaN   0.997    0.998  
punct_split               NaN   1.000    0.998  
zero_width                NaN   1.000    0.998

## 5. How well does the normaliser undo each family?

In-domain test set. The comparison target is the *normalised* clean text, i.e. what a detector
sees when the input was never obfuscated.

* `restored_exact`: the whole text matches (strict: one wrong character counts as a failure).
* `words_restored`: share of words recovered exactly; `words_intact_before` is the same measure
  before normalisation, so the gap between them is what the normaliser wins back.
* The NotInject row shows how often the normaliser leaves a benign prompt unchanged; the rest
  are prompts it alters (e.g. whitespace, "COVID-19" → "COVIDig"), which can cause false alarms.

In [10]:
nrep = evaluate.normaliser_report(bundle)
nrep.to_csv(TABLES / "normaliser_report.csv", index=False)
nrep

,family,n,obfuscated_share,restored_exact,words_restored,words_intact_before
0,clean,372,0.000,1.000,1.000,1.000
1,zero_width,372,1.000,1.000,1.000,0.587
2,homoglyph,372,1.000,0.524,0.938,0.596
3,diacritics,372,1.000,1.000,1.000,0.583
4,leetspeak,372,0.997,0.484,0.934,0.592
5,char_spacing,372,1.000,0.220,0.712,0.583
6,punct_split,372,1.000,1.000,1.000,0.604
7,NotInject benign: left unchanged,339,0.000,0.844,NaN,NaN


In [11]:
for t in ["COVID-19 cases rose in 2024.", "e.g. a well-known Zürich café", "Plan A or B: a b c"]:
    print(f"{t!r:40} -> {norm(t)!r}")

'COVID-19 cases rose in 2024.'           -> 'COVIDig cases rose in 2024.'
'e.g. a well-known Zürich café'          -> 'eg. a wellknown Zurich cafe'
'Plan A or B: a b c'                     -> 'Plan A or B: abc'
